# 1.3 Train a tiny language model

Read the matching lesson in `modules/01-standalone-foundations/03-learning-and-generation.md`. This notebook adds an executable walkthrough; the lesson supplies the full conceptual and organisational explanation.

**How to work:** select the project `.venv` kernel, then run cells from top to bottom. A kernel is the Python process that remembers values between cells. Restart it and run all cells to check that an experiment does not depend on hidden earlier work. Predict results before running a cell. Exercises and worked answers are labelled separately.

All data here are fictional. No API key, provider account or network call is used.


## Load the shared implementation

`import` makes saved functions available in this session. The editable course package locates the checkout, so this cell works even when Jupyter starts in another directory. `sys.path` tells Python where to find the pilot file. The functions themselves are fully explained in the notes and can be opened in `examples/01-foundations/foundations_lab.py`.


In [ ]:
from ai_orgdesign.workspace import project_root
import sys

ROOT = project_root()
LAB_FOLDER = ROOT / "examples" / "01-foundations"
if str(LAB_FOLDER) not in sys.path:
    sys.path.insert(0, str(LAB_FOLDER))
import foundations_lab as lab

print("Course code loaded. This notebook makes no network calls.")


## Inspect tokens and learned counts

Training counts which token follows another. `<START>` and `<END>` mark sentence boundaries. This bigram model conditions on just one preceding token. It cannot use the full delivery case or learn a modern neural representation.


In [ ]:
print(lab.tokenize("Check the order"))
counts = lab.train(lab.TRAINING_TEXT)
print(dict(counts["the"]))
print(lab.probabilities(counts, "the"))


The four training sentences contain two continuations to `order`, one to `record` and one to `evidence` after `the`. At temperature one, the conditional probabilities are therefore one half, one quarter and one quarter. An assertion checks a specific expectation; a failed assertion raises an error.


In [ ]:
distribution = lab.probabilities(counts, "the")
assert distribution == {"evidence": 0.25, "order": 0.5, "record": 0.25}
assert sum(distribution.values()) == 1.0
print(lab.generate(counts))
print(lab.generate(counts, max_tokens=2))


## Change the evidence used for learning

Creating a new list leaves the original training data intact. Adding repeated sentences changes the estimated distribution. Greedy decoding chooses the most probable continuation; equal probabilities use alphabetical tie-breaking in this implementation.


In [ ]:
changed_training = lab.TRAINING_TEXT + ["check the evidence"] * 3
changed_counts = lab.train(changed_training)
print(lab.probabilities(changed_counts, "the"))
print(lab.generate(changed_counts))
assert lab.generate(changed_counts)["text"] == "check the evidence"


## Exercise

What happens when the previous token has never appeared in training? Why should the notebook show that boundary explicitly?

## Worked answer

This implementation raises `ValueError`. Smoothing could allocate probability to unseen transitions, but would require a deliberate additional rule. Silently fabricating a learned distribution would misrepresent what this model estimated. The `try`/`except` block below catches the expected failure so the remaining cells can continue.


In [ ]:
try:
    lab.probabilities(counts, "invoice")
except ValueError as error:
    print("Expected boundary:", error)
else:
    raise AssertionError("An unknown context should fail.")
